# Modelamiento supervisado — riesgo crediticio

Entrena y compara modelos para predecir `Pago_atiempo`. Los datos llegan del
componente `src/ft_engineering.py` (limpieza, features y split ya definidos allí).
Por el desbalance ~20:1, la evaluación se centra en la clase minoritaria (no pago):
recall, precisión, F1, AUC-ROC y AUC-PR; el accuracy se reporta solo como referencia.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

raiz = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(raiz))

from src.ft_engineering import (cargar_datos, limpiar, crear_features,
                                construir_preprocesador, split_datos, OBJETIVO)

df = crear_features(limpiar(cargar_datos()))
X_train, X_test, y_train, y_test = split_datos(df)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)


def build_model(estimador, X_train, y_train):
    """Arma el pipeline preprocesador + estimador y lo entrena.

    El preprocesador se ajusta DENTRO del pipeline, solo con train:
    la imputacion y el escalado no ven nunca el test (sin fuga de datos).
    """
    modelo = Pipeline([
        ("preprocesador", construir_preprocesador(X_train)),
        ("clasificador", estimador),
    ])
    modelo.fit(X_train, y_train)
    return modelo


def summarize_classification(modelo, X_test, y_test, nombre="modelo"):
    """Calcula las metricas de evaluacion sobre la clase minoritaria (no pago = 0).

    pos_label=0: el evento de interes es NO pagar a tiempo, el raro (~5%).
    Es la misma logica de sensibilidad/especificidad con el evento como positivo.
    """
    y_pred = modelo.predict(X_test)
    # probabilidad de la clase 0 (no pago) para las AUC
    proba_no_pago = modelo.predict_proba(X_test)[:, list(modelo.classes_).index(0)]

    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_no_pago": precision_score(y_test, y_pred, pos_label=0),
        "recall_no_pago": recall_score(y_test, y_pred, pos_label=0),
        "f1_no_pago": f1_score(y_test, y_pred, pos_label=0),
        "auc_roc": roc_auc_score(y_test == 0, proba_no_pago),
        "auc_pr": average_precision_score(y_test == 0, proba_no_pago),
    }

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = build_model(
    LogisticRegression(max_iter=2000, class_weight="balanced"),
    X_train, y_train)

resultado_lr = summarize_classification(log_reg, X_test, y_test, "Regresión logística")
pd.DataFrame([resultado_lr]).round(4)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(log_reg, X_test, y_test)
plt.title("Regresión logística (balanceada) — matriz de confusión")
plt.show()